In [1]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

# list input files (optional sanity check)
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))




2026-05-06 05:33:49.945708: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778045629.960076      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778045629.964535      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-06 05:33:49.981554: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/kaggle/input/test.csv.zip
/kaggle/input/train.csv.zip
/kaggle/input/description.md
/kaggle/input/sample_submission.csv
/kaggle/input/train.csv
/kaggle/input/test.csv
/kaggle/input/sample_submission.csv.zip
/kaggle/input/tabular-playground-series-may-2022/test.csv.zip
/kaggle/input/tabular-playground-series-may-2022/train.csv.zip
/kaggle/input/tabular-playground-series-may-2022/description.md
/kaggle/input/tabular-playground-series-may-2022/sample_submission.csv
/kaggle/input/tabular-playground-series-may-2022/train.csv
/kaggle/input/tabular-playground-series-may-2022/test.csv
/kaggle/input/tabular-playground-series-may-2022/sample_submission.csv.zip


In [2]:
train_path = "/kaggle/input/tabular-playground-series-may-2022/train.csv"
test_path = "/kaggle/input/tabular-playground-series-may-2022/test.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

train.info()




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800000 entries, 0 to 799999
Data columns (total 33 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   id      800000 non-null  int64  
 1   f_00    800000 non-null  float64
 2   f_01    800000 non-null  float64
 3   f_02    800000 non-null  float64
 4   f_03    800000 non-null  float64
 5   f_04    800000 non-null  float64
 6   f_05    800000 non-null  float64
 7   f_06    800000 non-null  float64
 8   f_07    800000 non-null  int64  
 9   f_08    800000 non-null  int64  
 10  f_09    800000 non-null  int64  
 11  f_10    800000 non-null  int64  
 12  f_11    800000 non-null  int64  
 13  f_12    800000 non-null  int64  
 14  f_13    800000 non-null  int64  
 15  f_14    800000 non-null  int64  
 16  f_15    800000 non-null  int64  
 17  f_16    800000 non-null  int64  
 18  f_17    800000 non-null  int64  
 19  f_18    800000 non-null  int64  
 20  f_19    800000 non-null  float64
 21  f_20    80

In [3]:
# prepare feature matrix – drop identifier and target (and f_27 if it exists)
drop_cols = ["id", "target"]
if "f_27" in train.columns:
    drop_cols.append("f_27")
X_train = train.drop(columns=drop_cols)

scaler = StandardScaler()
scaled_train = scaler.fit_transform(X_train)




In [4]:
model = Sequential(
    [
        Dense(64, activation="relu", input_shape=(scaled_train.shape[1],)),
        Dense(32, activation="relu"),
        Dense(1, activation="sigmoid"),
    ]
)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(name="auc")],
)

callbacks = [
    EarlyStopping(patience=5, monitor="val_auc", mode="max", restore_best_weights=True)
]

history = model.fit(
    x=scaled_train,
    y=train["target"],
    validation_split=0.2,
    epochs=20,
    batch_size=64,
    callbacks=callbacks,
    verbose=2,
)




/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-05-06 05:34:04.384662: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1778045644.385114      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:45:00.0, compute capability: 8.6


Epoch 1/20


I0000 00:00:1778045645.637363     110 service.cc:148] XLA service 0x7f9c2800bc60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778045645.637395     110 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-05-06 05:34:05.655174: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778045645.744652     110 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1778045646.804383     110 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


10000/10000 - 13s - 1ms/step - auc: 0.8709 - loss: 0.4527 - val_auc: 0.9013 - val_loss: 0.4006
Epoch 2/20
10000/10000 - 12s - 1ms/step - auc: 0.9091 - loss: 0.3855 - val_auc: 0.9125 - val_loss: 0.3822
Epoch 3/20
10000/10000 - 12s - 1ms/step - auc: 0.9177 - loss: 0.3676 - val_auc: 0.9209 - val_loss: 0.3608
Epoch 4/20
10000/10000 - 11s - 1ms/step - auc: 0.9233 - loss: 0.3548 - val_auc: 0.9252 - val_loss: 0.3510
Epoch 5/20
10000/10000 - 11s - 1ms/step - auc: 0.9276 - loss: 0.3444 - val_auc: 0.9279 - val_loss: 0.3449
Epoch 6/20
10000/10000 - 11s - 1ms/step - auc: 0.9312 - loss: 0.3350 - val_auc: 0.9318 - val_loss: 0.3338
Epoch 7/20
10000/10000 - 11s - 1ms/step - auc: 0.9344 - loss: 0.3263 - val_auc: 0.9348 - val_loss: 0.3257
Epoch 8/20
10000/10000 - 11s - 1ms/step - auc: 0.9367 - loss: 0.3198 - val_auc: 0.9362 - val_loss: 0.3218
Epoch 9/20
10000/10000 - 12s - 1ms/step - auc: 0.9386 - loss: 0.3143 - val_auc: 0.9396 - val_loss: 0.3113
Epoch 10/20
10000/10000 - 12s - 1ms/step - auc: 0.9401 - 

In [5]:
# prepare test features with same columns as training
test_features = test.drop(columns=["id"])
if "f_27" in test_features.columns:
    test_features = test_features.drop(columns=["f_27"])
scaled_test = scaler.transform(test_features)




In [6]:
preds = model.predict(scaled_test, batch_size=64).flatten()  # probabilities




1563/1563 ━━━━━━━━━━━━━━━━━━━━ 1s 717us/step


In [7]:
submission = pd.DataFrame({"id": test["id"], "target": preds})
submission.to_csv("submit.csv", index=False)